# ARGUS — Master Colab Notebook

**Run cells top-to-bottom every session.** Steps 0–2 are fast (seconds). Step 3 only runs once (~24 GB). Step 4 is the long training run.

| Step | What it does | Time |
|------|-------------|------|
| 0 | Mount Drive + check GPU | 10 s |
| 1 | Clone / pull latest code | 10 s |
| 2 | Install packages | 2 min |
| 3 | Download datasets (first run only) | ~30 min |
| 4 | Train all models | hours |

> **Runtime → Change runtime type → GPU (A100 or T4)**

---
## Step 0 — Mount Drive & Check GPU

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, torch
print(f'GPU  : {torch.cuda.get_device_name(0)}')
print(f'VRAM : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')
print(f'Drive: {os.path.exists("/content/drive/MyDrive/ARGUS")}')

---
## Step 1 — Clone / Pull Repository

In [ ]:
import os

REPO = '/content/Argus'

if not os.path.exists(REPO):
    print('Cloning repo...')
    !git clone https://github.com/meteorboyF/Argus.git {REPO}
else:
    print('Pulling latest...')
    !cd {REPO} && git pull origin main

print('Done. Repo at:', REPO)

---
## Step 2 — Install Packages

Only needs to run once per session. Takes ~2 minutes.

In [ ]:
import subprocess, sys

def pip(*pkgs):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *pkgs])

print('Installing core packages...')
pip('torch', 'torchvision', 'torchaudio')
pip('opencv-python-headless', 'Pillow', 'albumentations', 'timm', 'einops', 'kornia')
pip('ultralytics', 'transformers', 'datasets', 'accelerate', 'peft', 'bitsandbytes', 'trl')
pip('insightface', 'easyocr', 'onnxruntime')
pip('openai-whisper', 'SpeechRecognition', 'pyaudio')
pip('huggingface_hub', 'gdown', 'wandb', 'tqdm', 'scipy', 'scikit-learn')
print('All packages ready.')

---
## Step 3 — Download Datasets & Models

**First run:** ~30 min, ~24 GB to Google Drive.  
**Subsequent runs:** skips everything already downloaded (seconds).

In [ ]:
!python /content/Argus/download_datasets.py

---
## Step 4 — Train All Models

Runs the full pipeline: stereo depth → segmentation → detection → privacy filter → speech → LLM → integration test.  
Each model saves checkpoints to Drive and resumes automatically if interrupted.

In [ ]:
!python /content/Argus/run_argus_pipeline.py

---
## Step 5 — Check Status (optional)

Run any time to see what's on Drive.

In [ ]:
import os
from pathlib import Path

BASE   = Path('/content/drive/MyDrive/ARGUS')
MODELS = BASE / 'models'
DS     = BASE / 'datasets'

def gb(p):
    p = Path(p)
    if not p.exists(): return 0
    return p.stat().st_size / 1e9 if p.is_file() else sum(f.stat().st_size for f in p.rglob('*') if f.is_file()) / 1e9

checks = [
    ('ADE20K',          DS/'ade20k'/'ade20k_done.flag',                         'segformer training'),
    ('NYUv2',           DS/'nyuv2'/'nyu_depth_v2_labeled.mat',                   'segformer supplemental'),
    ('COCO 2017',       DS/'coco'/'coco_done.flag',                              'yolov8 training'),
    ('RAFT-Stereo',     MODELS/'depth'/'raftstereo-realtime.pth',                'depth model'),
    ('SCRFD buffalo_s', MODELS/'privacy'/'models'/'buffalo_s'/'det_500m.onnx',   'face detection'),
    ('EasyOCR',         MODELS/'privacy'/'easyocr'/'easyocr_done.flag',          'text detection'),
    ('Phi-3.5 GGUF',    MODELS/'llm'/'Phi-3.5-mini-instruct-Q4_K_M.gguf',       'LLM'),
    ('Piper TTS',       MODELS/'speech'/'piper'/'en_US-lessac-medium.onnx',      'TTS'),
]

print('=' * 55)
print('  ARGUS Status')
print('=' * 55)
for label, path, use in checks:
    ok   = Path(path).exists()
    icon = '✅' if ok else '❌'
    print(f'  {icon}  {label:<22} — {use}')

total = gb(BASE)
print(f'\n  Drive used: {total:.1f} GB')
print('=' * 55)

---
## Step 6 — Run Individual Training Notebooks (optional)

If you want to re-run or debug a specific model, use these cells.

In [ ]:
# Stereo Depth (RAFT-Stereo)
!jupyter nbconvert --to notebook --execute /content/Argus/notebooks/01_stereo_depth.ipynb --inplace --ExecutePreprocessor.timeout=-1

In [ ]:
# Segmentation (SegFormer)
!jupyter nbconvert --to notebook --execute /content/Argus/notebooks/02_segmentation.ipynb --inplace --ExecutePreprocessor.timeout=-1

In [ ]:
# Object Detection (YOLOv8)
!jupyter nbconvert --to notebook --execute /content/Argus/notebooks/03_object_detection.ipynb --inplace --ExecutePreprocessor.timeout=-1

In [ ]:
# Privacy Filter (face + text blurring)
!jupyter nbconvert --to notebook --execute /content/Argus/notebooks/04_privacy_filter.ipynb --inplace --ExecutePreprocessor.timeout=-1

In [ ]:
# Speech Pipeline (Whisper + Piper + Wake Word)
!jupyter nbconvert --to notebook --execute /content/Argus/notebooks/05_speech_pipeline.ipynb --inplace --ExecutePreprocessor.timeout=-1

In [ ]:
# LLM Setup (Phi-3.5 + optional LoRA fine-tune)
!jupyter nbconvert --to notebook --execute /content/Argus/notebooks/06_llm_setup.ipynb --inplace --ExecutePreprocessor.timeout=-1

In [ ]:
# Integration Test
!jupyter nbconvert --to notebook --execute /content/Argus/notebooks/07_integration_test.ipynb --inplace --ExecutePreprocessor.timeout=-1